In [ ]:
# ============================================================
# GPR prediction of Aging Index
# Mass-based inputs + Leave-One-Mastic-Out CV
#
# Input file:
#   /content/Aging index.xlsx
#
# Inputs:
#   Filler type
#   F/B mass
#   log10(Frequency)
#
# Target:
#   Aging Index
#
# Validation:
#   Nested Leave-One-Mastic-Out Cross-Validation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import joblib

from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    Matern,
    RationalQuadratic,
    ConstantKernel as C,
    WhiteKernel
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# 1. Read Excel file
# ------------------------------------------------------------

file_path = "/content/Aging index.xlsx"

data = pd.read_excel(file_path)
data.columns = data.columns.astype(str).str.strip()

print("Columns in file:")
print(data.columns.tolist())


# ------------------------------------------------------------
# 2. Rename possible column names
# ------------------------------------------------------------

rename_map = {
    "Filler type": "Filler_type",
    "Filler Type": "Filler_type",
    "Filler_type": "Filler_type",

    "F/B mass": "FB_mass",
    "FB mass": "FB_mass",
    "FB_mass": "FB_mass",
    "f/b by weight": "FB_mass",
    "F/B by weight": "FB_mass",
    "F/B By Weight": "FB_mass",

    "Frequency (rad/s)": "Frequency",
    "Frequency": "Frequency",
    "Frequency (rad/sec)": "Frequency",

    "Aging Index": "Aging_Index",
    "Aging index": "Aging_Index",
    "Aging_Index": "Aging_Index",
    "AI": "Aging_Index",
    "AI @ freq.": "Aging_Index"
}

data = data.rename(columns=rename_map)


# ------------------------------------------------------------
# 3. Automatic column detection if names differ
# ------------------------------------------------------------

def find_column(df, accepted_names, keywords):

    for name in accepted_names:
        if name in df.columns:
            return name

    for col in df.columns:
        col_lower = str(col).lower()

        if all(
            keyword.lower() in col_lower
            for keyword in keywords
        ):
            return col

    return None


filler_col = find_column(
    data,
    ["Filler_type"],
    ["filler"]
)

fbmass_col = find_column(
    data,
    ["FB_mass"],
    ["weight"]
)

if fbmass_col is None:
    fbmass_col = find_column(
        data,
        ["FB_mass"],
        ["mass"]
    )

frequency_col = find_column(
    data,
    ["Frequency"],
    ["frequency"]
)

aging_col = find_column(
    data,
    ["Aging_Index"],
    ["aging", "index"]
)

if aging_col is None:
    aging_col = find_column(
        data,
        ["Aging_Index"],
        ["ai"]
    )

detected_columns = {
    "Filler_type": filler_col,
    "FB_mass": fbmass_col,
    "Frequency": frequency_col,
    "Aging_Index": aging_col
}

print("\nDetected columns:")
print(detected_columns)

missing = [
    key
    for key, value in detected_columns.items()
    if value is None
]

if missing:
    raise ValueError(
        f"Could not detect required columns: {missing}"
    )

data = data.rename(columns={
    filler_col: "Filler_type",
    fbmass_col: "FB_mass",
    frequency_col: "Frequency",
    aging_col: "Aging_Index"
})


# ------------------------------------------------------------
# 4. Keep and clean required columns
# ------------------------------------------------------------

required_cols = [
    "Filler_type",
    "FB_mass",
    "Frequency",
    "Aging_Index"
]

data = data[required_cols].copy()

data["Filler_type"] = (
    data["Filler_type"]
    .astype(str)
    .str.strip()
)

for col in [
    "FB_mass",
    "Frequency",
    "Aging_Index"
]:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

data = data.dropna().reset_index(drop=True)

data = data[
    (data["Frequency"] > 0) &
    (data["Aging_Index"] > 0)
].reset_index(drop=True)


# ------------------------------------------------------------
# 5. Normalize filler names
# ------------------------------------------------------------

filler_name_map = {
    "Rh": "Rheofiller",
    "RH": "Rheofiller",
    "rheofiller": "Rheofiller",

    "Ce": "Cement",
    "CE": "Cement",
    "cement": "Cement",

    "Ch": "Chromium steel sludge",
    "CH": "Chromium steel sludge",
    "chromium steel sludge": "Chromium steel sludge"
}

data["Filler_type"] = data["Filler_type"].replace(
    filler_name_map
)

print("\nUnique filler names:")
print(data["Filler_type"].unique())


# ------------------------------------------------------------
# 6. Transform frequency
# ------------------------------------------------------------

data["log10_Frequency"] = np.log10(
    data["Frequency"]
)

# Aging Index remains on its original scale.


# ------------------------------------------------------------
# 7. Create mastic groups
#
# Each group represents one complete mastic formulation.
# All four frequencies of one mastic are removed together.
# ------------------------------------------------------------

data["Mastic_Group"] = (
    data["Filler_type"].astype(str)
    + "_FBmass_"
    + data["FB_mass"].round(3).astype(str)
)

print("\nClean Aging Index dataset:")
display(data)

print("\nNumber of observations:", len(data))

print(
    "Number of mastic groups:",
    data["Mastic_Group"].nunique()
)

print("\nMastic groups:")

display(
    data[
        [
            "Filler_type",
            "FB_mass",
            "Mastic_Group"
        ]
    ].drop_duplicates()
)

group_size_check = (
    data.groupby("Mastic_Group")
    .size()
    .reset_index(name="Number_of_rows")
)

print("\nRows in each mastic group:")
display(group_size_check)


# ------------------------------------------------------------
# 8. Define inputs, target, and groups
# ------------------------------------------------------------

X_cols = [
    "Filler_type",
    "FB_mass",
    "log10_Frequency"
]

X = data[X_cols]

y = data["Aging_Index"].values

groups = data["Mastic_Group"].values


# ------------------------------------------------------------
# 9. Preprocessing
# ------------------------------------------------------------

categorical_features = [
    "Filler_type"
]

numerical_features = [
    "FB_mass",
    "log10_Frequency"
]

filler_categories = [[
    "Chromium steel sludge",
    "Cement",
    "Rheofiller"
]]

try:
    encoder = OneHotEncoder(
        categories=filler_categories,
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:
    encoder = OneHotEncoder(
        categories=filler_categories,
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            encoder,
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ],
    remainder="drop"
)

# 3 filler columns + 2 numerical variables
n_features_after_preprocessing = 5


# ------------------------------------------------------------
# 10. Candidate GPR kernels
# ------------------------------------------------------------

kernel_options = [

    C(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    * RBF(
        length_scale=np.ones(
            n_features_after_preprocessing
        ),
        length_scale_bounds=(1e-3, 1e3)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-8, 1e1)
    ),

    C(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    * Matern(
        length_scale=np.ones(
            n_features_after_preprocessing
        ),
        length_scale_bounds=(1e-3, 1e3),
        nu=1.5
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-8, 1e1)
    ),

    C(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    * RationalQuadratic(
        length_scale=1.0,
        alpha=1.0,
        length_scale_bounds=(1e-3, 1e3),
        alpha_bounds=(1e-3, 1e3)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-8, 1e1)
    )
]

param_grid = {
    "regressor__gpr__kernel": kernel_options
}


# ------------------------------------------------------------
# 11. Function for creating the GPR model
# ------------------------------------------------------------

def make_gpr_model():

    gpr = GaussianProcessRegressor(
        normalize_y=False,
        n_restarts_optimizer=20,
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("gpr", gpr)
    ])

    model = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=StandardScaler()
    )

    return model


# ------------------------------------------------------------
# 12. Nested Leave-One-Mastic-Out CV
#
# Outer CV:
#   evaluates one completely unseen mastic
#
# Inner CV:
#   selects the best GPR kernel using training mastics only
# ------------------------------------------------------------

outer_cv = LeaveOneGroupOut()

y_true_all = []
y_pred_all = []

test_indices_all = []
heldout_groups_all = []
best_params_all = []

fold_number = 1

for train_idx, test_idx in outer_cv.split(
    X,
    y,
    groups
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    groups_train = groups[train_idx]

    heldout_group = np.unique(
        groups[test_idx]
    )[0]

    print(
        f"\nOuter fold {fold_number}: "
        f"holding out {heldout_group}"
    )

    model = make_gpr_model()

    inner_cv = LeaveOneGroupOut()

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        n_jobs=-1,
        return_train_score=True
    )

    grid.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    best_model_fold = grid.best_estimator_

    y_pred_fold = best_model_fold.predict(
        X_test
    )

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred_fold)

    test_indices_all.extend(test_idx)

    heldout_groups_all.extend(
        [heldout_group] * len(test_idx)
    )

    best_params_all.extend(
        [str(grid.best_params_)] * len(test_idx)
    )

    print("Best kernel in this fold:")
    print(grid.best_params_)

    fold_number += 1


# ------------------------------------------------------------
# 13. Restore original row order
# ------------------------------------------------------------

y_true_all = np.asarray(y_true_all)
y_pred_all = np.asarray(y_pred_all)

test_indices_all = np.asarray(
    test_indices_all
)

heldout_groups_all = np.asarray(
    heldout_groups_all
)

best_params_all = np.asarray(
    best_params_all
)

order = np.argsort(test_indices_all)

y_true_all = y_true_all[order]
y_pred_all = y_pred_all[order]

test_indices_all = test_indices_all[order]

heldout_groups_all = (
    heldout_groups_all[order]
)

best_params_all = best_params_all[order]


# ------------------------------------------------------------
# 14. Metric functions
# ------------------------------------------------------------

def rmse_score(y_true, y_pred):

    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


def mape_score(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    nonzero = y_true != 0

    if np.sum(nonzero) == 0:
        return np.nan

    return np.mean(
        np.abs(
            (
                y_true[nonzero] -
                y_pred[nonzero]
            ) /
            y_true[nonzero]
        )
    ) * 100


def smape_score(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    denominator = (
        np.abs(y_true) +
        np.abs(y_pred)
    )

    valid = denominator != 0

    if np.sum(valid) == 0:
        return np.nan

    return np.mean(
        2
        * np.abs(
            y_pred[valid] -
            y_true[valid]
        )
        / denominator[valid]
    ) * 100


# ------------------------------------------------------------
# 15. Overall LOMO metrics
# ------------------------------------------------------------

r2 = r2_score(
    y_true_all,
    y_pred_all
)

rmse = rmse_score(
    y_true_all,
    y_pred_all
)

mae = mean_absolute_error(
    y_true_all,
    y_pred_all
)

mape = mape_score(
    y_true_all,
    y_pred_all
)

smape = smape_score(
    y_true_all,
    y_pred_all
)

metrics = pd.DataFrame([{
    "Validation": "Leave-One-Mastic-Out",
    "Model": "GPR",
    "Target": "Aging_Index",
    "R2": r2,
    "RMSE": rmse,
    "MAE": mae,
    "MAPE_%": mape,
    "sMAPE_%": smape
}])

print(
    "\nGPR Leave-One-Mastic-Out "
    "metrics for Aging Index:"
)

display(metrics)


# ------------------------------------------------------------
# 16. Prediction table
# ------------------------------------------------------------

predictions = (
    data.iloc[test_indices_all]
    .copy()
    .reset_index(drop=True)
)

predictions["Heldout_Group"] = (
    heldout_groups_all
)

predictions["Actual_Aging_Index"] = (
    y_true_all
)

predictions["Predicted_Aging_Index"] = (
    y_pred_all
)

predictions["Residual"] = (
    y_true_all -
    y_pred_all
)

predictions["Absolute_Error"] = np.abs(
    predictions["Residual"]
)

predictions["Error_%"] = (
    predictions["Residual"] /
    predictions["Actual_Aging_Index"]
) * 100

predictions["Absolute_Percentage_Error_%"] = (
    np.abs(
        predictions["Error_%"]
    )
)

predictions["Best_params_in_fold"] = (
    best_params_all
)

print("\nLOMO predictions:")
display(predictions)


# ------------------------------------------------------------
# 17. Metrics for each held-out mastic
# ------------------------------------------------------------

group_metrics_rows = []

for group_name, group_df in predictions.groupby(
    "Mastic_Group"
):

    y_group_true = group_df[
        "Actual_Aging_Index"
    ].values

    y_group_pred = group_df[
        "Predicted_Aging_Index"
    ].values

    group_metrics_rows.append({
        "Mastic_Group": group_name,

        "Filler_type":
            group_df["Filler_type"].iloc[0],

        "FB_mass":
            group_df["FB_mass"].iloc[0],

        "R2":
            r2_score(
                y_group_true,
                y_group_pred
            ),

        "RMSE":
            rmse_score(
                y_group_true,
                y_group_pred
            ),

        "MAE":
            mean_absolute_error(
                y_group_true,
                y_group_pred
            ),

        "MAPE_%":
            mape_score(
                y_group_true,
                y_group_pred
            ),

        "sMAPE_%":
            smape_score(
                y_group_true,
                y_group_pred
            )
    })

group_metrics = pd.DataFrame(
    group_metrics_rows
)

print("\nErrors for each held-out mastic:")
display(group_metrics)


# ------------------------------------------------------------
# 18. Actual versus predicted plot
# ------------------------------------------------------------

plot_min = min(
    y_true_all.min(),
    y_pred_all.min()
)

plot_max = max(
    y_true_all.max(),
    y_pred_all.max()
)

margin = 0.05 * (
    plot_max - plot_min
)

plt.figure(figsize=(6, 6))

plt.scatter(
    y_true_all,
    y_pred_all,
    s=55
)

plt.plot(
    [
        plot_min - margin,
        plot_max + margin
    ],
    [
        plot_min - margin,
        plot_max + margin
    ],
    linewidth=1.5
)

plt.xlabel("Actual Aging Index")
plt.ylabel("Predicted Aging Index")

plt.title(
    "GPR Leave-One-Mastic-Out: Aging Index"
)

plt.xlim(
    plot_min - margin,
    plot_max + margin
)

plt.ylim(
    plot_min - margin,
    plot_max + margin
)

plt.grid(True, alpha=0.4)
plt.tight_layout()

plt.savefig(
    "GPR_AgingIndex_LOMO_actual_vs_predicted.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 19. Residual plot
# ------------------------------------------------------------

residuals = (
    y_true_all -
    y_pred_all
)

plt.figure(figsize=(6, 4))

plt.scatter(
    y_pred_all,
    residuals,
    s=55
)

plt.axhline(
    0,
    linewidth=1.5
)

plt.xlabel("Predicted Aging Index")
plt.ylabel("Residual")

plt.title(
    "GPR Leave-One-Mastic-Out residuals"
)

plt.grid(True, alpha=0.4)
plt.tight_layout()

plt.savefig(
    "GPR_AgingIndex_LOMO_residuals.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 20. Experimental and predicted AI versus frequency
# ------------------------------------------------------------

plot_df = predictions.sort_values(
    [
        "Filler_type",
        "FB_mass",
        "Frequency"
    ]
)

for (
    filler,
    fb_mass
), subset in plot_df.groupby(
    [
        "Filler_type",
        "FB_mass"
    ]
):

    subset = subset.sort_values(
        "Frequency"
    )

    plt.figure(figsize=(6, 4))

    plt.plot(
        subset["Frequency"],
        subset["Actual_Aging_Index"],
        marker="o",
        label="Experimental"
    )

    plt.plot(
        subset["Frequency"],
        subset["Predicted_Aging_Index"],
        marker="s",
        linestyle="--",
        label="GPR prediction"
    )

    plt.xscale("log")

    plt.xlabel(
        "Reduced angular frequency (rad/s)"
    )

    plt.ylabel("Aging Index")

    plt.title(
        f"{filler}, F/B mass = {fb_mass}"
    )

    plt.grid(
        True,
        which="both",
        alpha=0.4
    )

    plt.legend()
    plt.tight_layout()

    safe_filler = (
        filler.replace(" ", "_")
        .replace("/", "_")
    )

    plt.savefig(
        f"GPR_AI_curve_{safe_filler}_"
        f"FBmass_{fb_mass}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ------------------------------------------------------------
# 21. Select final GPR model using all data
# ------------------------------------------------------------

final_model = make_gpr_model()

final_grid = GridSearchCV(
    estimator=final_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=LeaveOneGroupOut(),
    n_jobs=-1,
    return_train_score=True
)

final_grid.fit(
    X,
    y,
    groups=groups
)

best_final_model = (
    final_grid.best_estimator_
)

best_final_params = (
    final_grid.best_params_
)

print("\nBest final GPR parameters:")
print(best_final_params)


# ------------------------------------------------------------
# 22. Grid-search results
# ------------------------------------------------------------

grid_results = pd.DataFrame(
    final_grid.cv_results_
)

grid_results_clean = grid_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "param_regressor__gpr__kernel"
    ]
].copy()

grid_results_clean = (
    grid_results_clean
    .sort_values("rank_test_score")
)

grid_results_clean[
    "RMSE_grouped_CV"
] = -grid_results_clean[
    "mean_test_score"
]

print("\nGrouped GPR grid-search results:")
display(grid_results_clean)


# ------------------------------------------------------------
# 23. Save final model
# ------------------------------------------------------------

gpr_framework = {
    "model": best_final_model,

    "X_cols": X_cols,

    "target": "Aging_Index",

    "best_params": best_final_params,

    "input_features": (
        "Filler type, F/B mass, "
        "log10(Frequency)"
    ),

    "validation": (
        "Nested Leave-One-Mastic-Out CV"
    )
}

joblib.dump(
    gpr_framework,
    "GPR_AgingIndex_FBmass_LOMO_final.pkl"
)


# ------------------------------------------------------------
# 24. Export results to Excel
# ------------------------------------------------------------

parity_data = pd.DataFrame({
    "Parity_x": [
        plot_min - margin,
        plot_max + margin
    ],
    "Parity_y": [
        plot_min - margin,
        plot_max + margin
    ]
})

plot_data = pd.DataFrame({
    "Actual_Aging_Index":
        y_true_all,

    "Predicted_Aging_Index":
        y_pred_all,

    "Residual":
        residuals
})

output_file = (
    "GPR_AgingIndex_FBmass_LOMO_results.xlsx"
)

with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    data.to_excel(
        writer,
        sheet_name="Clean_Data",
        index=False
    )

    metrics.to_excel(
        writer,
        sheet_name="Overall_Metrics",
        index=False
    )

    predictions.to_excel(
        writer,
        sheet_name="Predictions_LOMO",
        index=False
    )

    group_metrics.to_excel(
        writer,
        sheet_name="Group_Errors",
        index=False
    )

    grid_results_clean.to_excel(
        writer,
        sheet_name="Grid_Search",
        index=False
    )

    plot_data.to_excel(
        writer,
        sheet_name="Plot_Data",
        index=False
    )

    parity_data.to_excel(
        writer,
        sheet_name="Parity_Line",
        index=False
    )


print("\nFiles saved:")
print(
    "1. GPR_AgingIndex_FBmass_LOMO_results.xlsx"
)
print(
    "2. GPR_AgingIndex_LOMO_actual_vs_predicted.png"
)
print(
    "3. GPR_AgingIndex_LOMO_residuals.png"
)
print(
    "4. GPR_AgingIndex_FBmass_LOMO_final.pkl"
)